# imports 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType,DateType

# read date from Bronze Layer

In [0]:
df = spark.read.table('`e-commerce-databricks-project`.bronze_layer.crm_sales_details')


# apply Data Transformations for silver Table

In [0]:
df.display()

## Renaming columns

In [0]:

RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old,new in RENAME_MAP.items():
    df = df.withColumnRenamed(old,new)

## change customer_id column type to string

In [0]:
df = df.withColumn('customer_id', col('customer_id').cast(StringType()))

## Triming String columns

In [0]:
for colName, colType in df.dtypes:
    if colType == 'string':
        df = df.withColumn(colName,trim(col(colName)))
df.display()

## fixing Sales and Price 

In [0]:
df = (df.withColumn(
        "price",
       when(
            (col("price").isNull()) | (col("price") <= 0),
            when(
                col("quantity") != 0,
                col("sales_amount") / col("quantity")
            ).otherwise(None)
        ).otherwise(col("price"))
    )
)

## Fixing Dates


In [0]:
df = (
    df
    .withColumn(
        "order_date",
        when(
            (col("order_date") == 0) | (length(col("order_date")) != 8),
            None
        ).otherwise(to_date(col("order_date").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "ship_date",
        when(
            (col("ship_date") == 0) | (length(col("ship_date")) != 8),
            None
        ).otherwise(to_date(col("ship_date").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "due_date",
        when(
            (col("due_date") == 0) | (length(col("due_date")) != 8),
            None
        ).otherwise(to_date(col("due_date").cast("string"), "yyyyMMdd"))
    )
)

## Final Look

In [0]:
df.display()

# write data into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('`e-commerce-databricks-project`.silver_layer.Sales_Details')